# 07 · 噪声与纠错：用 3 个比特保护 1 个比特

对应讲义实验 F · 前置：01 编程基础

**问题**：真实的量子比特会出错。今天最好的硬件，两比特门的错误率约为 $10^{-3}$；一个有用的算法却需要上亿次操作。只靠改进硬件远远不够，必须纠错。

**思路**：用冗余换可靠。把 1 个逻辑比特编码到多个物理比特上，出错时少数服从多数。纠错的基本规律是：**只要物理错误率低于阈值，码距越大，逻辑错误越少**。

In [ ]:
from math import comb
import numpy as np
import matplotlib.pyplot as plt
from mindquantum import Circuit, H, X, RY, Simulator, Hamiltonian, QubitOperator
from mindquantum import BitFlipChannel, DepolarizingChannel, PhaseFlipChannel
from mqlab import check

## 1. 噪声：同一个 Bell 态，理想与含噪

`DepolarizingChannel(p)` 以概率 p 让比特变得完全随机（随机作用 I、X、Y、Z 之一）。在 CNOT 之后，给两个比特各加一个 p = 5% 的去极化信道：

In [ ]:
bell = Circuit([H.on(0), X.on(1, 0)])
noise = DepolarizingChannel(0.05)          # 以 5% 的概率完全随机化
noisy = bell + noise.on(0) + noise.on(1)
for name, c in [('理想', bell), ('含噪', noisy)]:
    c = c.copy()
    c.measure_all()
    res = Simulator('mqvector', 2).sampling(c, shots=1000, seed=42)
    print(name)
    display(res.svg())

含噪时出现了 `01` 和 `10`，两个比特不再总是一致。试一试：把 0.05 改成 0.3。

在态矢量后端 `mqvector` 上，噪声信道以“轨迹采样”的方式模拟：每次采样随机决定是否出错。

## 2. 重复码：编码、出错、多数表决

把一个比特复制成 3 份（$|1\rangle \to |111\rangle$），读出时少数服从多数：只要出错的比特不超过一个，就能得到正确答案。量子态不能复制（不可克隆定理），但可以用 CNOT 把 $\alpha|0\rangle + \beta|1\rangle$ 编码成 $\alpha|000\rangle + \beta|111\rangle$。

下面编码 $|1\rangle$，让每个比特独立地以概率 p 翻转（`BitFlipChannel`），再测量并多数表决：

In [ ]:
def repetition(d, p):
    circ = Circuit([X.on(0)])                  # 逻辑比特 |1⟩
    for i in range(1, d):
        circ += X.on(i, 0)                     # 编码：|1⟩ → |1…1⟩
    for i in range(d):
        circ += BitFlipChannel(p).on(i)        # 以概率 p 翻转
    circ.measure_all()
    return circ

def logical_error(d, p, shots=100_000, seed=1):
    sim = Simulator('mqvector', d)
    res = sim.sampling(repetition(d, p), shots=shots, seed=seed)
    wrong = sum(n for bits, n in res.data.items()
                if bits.count('1') <= d // 2)  # 多数表决出错
    return wrong / shots

先看一次实验：码距 3、p = 5%，采样 1,000 次。

In [ ]:
res = Simulator('mqvector', 3).sampling(repetition(3, 0.05), shots=1000, seed=1)
res.svg()

大部分结果是 `111`；出错的结果里，绝大多数只翻了一个比特（`110`、`101`、`011`），多数表决能把它们纠正。按 $p = 5\%$ 算一笔账：0 错占 $85.7\%$、1 错占 $13.5\%$（合计 $99.27\%$ 都能救回）；只有两个以上比特同时翻转（`100`、`010`、`001`、`000`，共 $0.73\%$）时，表决才会出错。

下面是完整的比较（与讲义相同）：同样的物理错误率，码距 1（不编码）与码距 3 的逻辑错误率。

In [ ]:
p = 0.05                                   # 物理错误率
for d in [1, 3]:                           # 码距
    print(f'd = {d}：逻辑错误率 {logical_error(d, p):.3g}')

### 现场改一改：先预测，再运行

直接修改上面的单元格，先写下预测再运行：

1. **物理错误率很高时**：改成 `p = 0.6`、`for d in [1, 3, 5]`。码距越大，逻辑错误会？　A 越少　B 差不多　C 越多
2. **码距 5**：再把 p 改回 `0.05`（保留 d = 5）。逻辑错误率约为？　A 1%　B 0.1%　C 0.01%
3. **换成相位翻转**：码距 3 能比码距 1 保护得更好吗？　A 能　B 一样　C 反而更差。相位翻转在计算基测量中看不见，要换一种测法，在第 5 节运行。

<details><summary>答案</summary>

1. C：0.601、0.648、0.684。每个比特都更可能出错时，多数表决反而更可能选错：高于阈值，越纠越错（见第 3 节的图）。
2. B：约 0.1%（实测 0.00098，理论 0.00116）。d = 1 时 5%，d = 3 时 0.77%，d = 5 时约 0.1%：每多 2 个比特，错误率降低约一个量级。
3. C：见第 5 节。
</details>

## 3. 深入：码距越大，抑制越强——前提是低于阈值

码距为 d 的重复码，只有**超过一半**的比特翻转时，多数表决才会出错：

$$P_L = \sum_{k > d/2} \binom{d}{k} p^k (1-p)^{d-k}.$$

比较 d = 1、3、5、7：点是模拟结果，线是这个公式。

In [ ]:
def theory(d, p):                          # 超过一半的比特翻转的概率
    return sum(comb(d, k) * p ** k * (1 - p) ** (d - k) for k in range(d // 2 + 1, d + 1))

ps = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
pp = np.linspace(0.03, 0.7, 100)
plt.figure(figsize=(6, 4))
for d, color in zip([1, 3, 5, 7], ['0.6', '#e59866', '#c0392b', '#641e16']):
    plt.semilogy(ps, [logical_error(d, p) for p in ps], 'o', color=color, label=f'd = {d}')
    plt.semilogy(pp, [theory(d, p) for p in pp], color=color, lw=1)
plt.axvline(0.5, color='0.5', ls='--', lw=1)
plt.xlabel('physical error rate p')
plt.ylabel('logical error rate')
plt.legend()
plt.show()

所有曲线在 p = 0.5 交于一点，这是这个简化模型的**阈值**：低于阈值时，增大码距能把错误率压到任意低；高于阈值（虚线右侧），纠错反而越纠越错。真实硬件上表面码的阈值约为 1%。2024 年，Google 的 Willow 芯片首次在表面码上实现了“码距 3 → 5 → 7、逻辑错误率逐级下降”，码距每增加 2，错误率降为约 1/2.14。

## 4. 症状与解码器：不看数据，也能找到错误

多数表决要直接测量数据比特，这会破坏量子叠加。真正的纠错只测量**奇偶校验**：

- $s_0 = q_0 \oplus q_1$，$s_1 = q_1 \oplus q_2$，用两个辅助比特 $q_3$、$q_4$ 和 CNOT 得到；
- 症状 $(s_0, s_1)$ 指出哪个比特出了错，却不泄露逻辑信息；
- 根据症状决定纠正哪个比特的经典程序，叫做**解码器**。

下面用 RY(1.0) 制备一个叠加态作为逻辑比特，编码后在不同位置注入一个比特翻转，再测症状：

In [ ]:
def encode_and_check(error_on=None):
    circ = Circuit([RY(1.0).on(0), X.on(1, 0), X.on(2, 0)])  # 编码
    if error_on is not None:
        circ += X.on(error_on)                  # 注入一个比特翻转
    circ += Circuit([X.on(3, 0), X.on(3, 1)])   # s0 = q0 ⊕ q1
    circ += Circuit([X.on(4, 1), X.on(4, 2)])   # s1 = q1 ⊕ q2
    return circ

for err in [None, 0, 1, 2]:
    sim = Simulator('mqvector', 5)
    sim.apply_circuit(encode_and_check(err))
    ket = sim.get_qs(ket=True).replace('\n', '  ')
    print(f'错误位置 {str(err):>4}：{ket}')

比特串按 $q_4 q_3 q_2 q_1 q_0$ 书写，前两位是症状 $(s_1, s_0)$。如果直接测数据比特，会把 $|010\rangle$ 与 $|101\rangle$ 区分开、当场破坏叠加；而只测相邻奇偶时，两项的症状完全相同——辅助比特“看不出”逻辑 0 或 1，测量症状不会让叠加坍缩，数据比特上的系数 0.88、0.48 原样保留。

### 练习 · 写一个解码器

补全 `decode(s0, s1)`：返回需要翻转的比特编号（0、1、2），不需要纠正时返回 `None`。

In [ ]:
def decode(s0, s1):
    # 练习
    return ...

check.decoder(decode)

把解码器接到线路上：读出症状，按解码结果纠正，再和没有出错的编码态比较。

In [ ]:
encoded = Simulator('mqvector', 3)
encoded.apply_circuit(Circuit([RY(1.0).on(0), X.on(1, 0), X.on(2, 0)]))
target = encoded.get_qs()                       # 没有出错的编码态

for err in [None, 0, 1, 2]:
    sim = Simulator('mqvector', 5)
    sim.apply_circuit(encode_and_check(err))
    idx = int(np.argmax(np.abs(sim.get_qs())))  # 症状是确定的，任取一项
    s0, s1 = (idx >> 3) & 1, (idx >> 4) & 1
    fix = decode(s0, s1)
    if fix is ...:
        print('请先完成练习')
        break
    if fix is not None:
        sim.apply_gate(X.on(fix))               # 纠正
    data = sim.get_qs().reshape(4, 8)[2 * s1 + s0]  # 数据比特的分量
    fid = abs(np.vdot(target, data)) ** 2
    print(f'错误 {str(err):>4} → 症状 ({s0}, {s1}) → 纠正 {fix}，保真度 {fid:.3f}')

## 5. 重复码能防相位翻转吗？

量子比特还有一种经典比特没有的错误：相位翻转 Z（$|+\rangle \leftrightarrow |-\rangle$）。它不改变计算基的测量结果，所以要换一种测法：把逻辑 $|+\rangle$ 编码为 $(|000\rangle + |111\rangle)/\sqrt{2}$，用逻辑算符 $X_0X_1X_2$ 的期望值衡量信息是否保存完好（没有错误时为 1）。这里用密度矩阵后端 `mqmatrix` 精确计算。

In [ ]:
p = 0.05
for d in [1, 3]:
    circ = Circuit([H.on(0)] + [X.on(i, 0) for i in range(1, d)])
    circ += Circuit([PhaseFlipChannel(p).on(i) for i in range(d)])
    logical_x = Hamiltonian(QubitOperator(' '.join(f'X{i}' for i in range(d))))
    value = Simulator('mqmatrix', d).get_expectation(logical_x, circ).real
    print(f'd = {d}   <X…X> = {value:.3f}')

码距 3 反而更差（现场改一改第 3 处的答案是 C）：对任意一个比特施加相位翻转 $Z$，都会把 $|111\rangle$ 变号——$Z_0(|000\rangle+|111\rangle) = Z_1(\cdots) = Z_2(\cdots) = |000\rangle - |111\rangle$。奇数个比特出 $Z$ 错就会翻转逻辑相位，理论期望值 $\langle X\cdots X\rangle = (1 - 2p)^d$：$p = 5\%$ 时 $d = 1$ 为 $0.900$，$d = 3$ 掉到 $0.9^3 = 0.729$，与上面的 `mqmatrix` 实测严丝合缝——冗余非但没帮忙，反而让出错机会翻了三倍。试一试：把 `PhaseFlipChannel` 换回 `BitFlipChannel`，两个值都是 1——比特翻转不影响这种测法。

要同时防住比特翻转和相位翻转，需要更强的码：Shor 码用 9 个比特做到了这一点；今天的主流是**表面码**，码距 d 需要约 $2d^2$ 个物理比特，症状每微秒测量一次，解码器必须实时跟上。

## 小结

- 噪声让结果偏离理想分布；纠错用冗余换取可靠性。
- 低于阈值时，码距越大，逻辑错误率越低——这是所有纠错方案的共同规律。
- 纠错的核心是软件：解码器要根据症状又快又准地推断错误，这是今天量子软件最活跃的方向之一。